### Bellow, necessary imports

In [2]:
import Sboxes_and_examples
from typing import List
import numpy as np

### AES utils Classes and functions

In [3]:
class AESUtils:
    def __init__(self):
        # Rcon values for AES
        self.Rcon = [
            0x01, 0x02, 0x04, 0x08, 
            0x10, 0x20, 0x40, 0x80, 
            0x1B, 0x36, 0x6C, 0xD8, 
            0xAB, 0x4D, 0x9A, 0x2F
        ]
        # Matrix mutltiplication matrix
        self.MM = np.array([
            [0x02, 0x03, 0x01, 0x01], 
            [0x01, 0x02, 0x03, 0x01], 
            [0x01, 0x01, 0x02, 0x03], 
            [0x03, 0x01, 0x01, 0x02]
        ])
        self.MM_Inv = np.array([
            [0x0E, 0x0B, 0x0D, 0x09], 
            [0x09, 0x0E, 0x0B, 0x0D], 
            [0x0D, 0x09, 0x0E, 0x0B], 
            [0x0B, 0x0D, 0x09, 0x0E]
        ])

    def print_hex(self, matrix):
        """ Prints matrix as hexadecimal matrix """
        for line in matrix:
            for elt in line:
                print(format(elt, '#04x').upper(), end=" ")
            print()
        print()
    
    
    def convert_text_to_bin(self, plain_text: str):
        """ convert text message to binary """
        bin_message = "".join([format(c, '08b') for c in plain_text.encode('utf-8')])
        return bin_message
    
    def convert_hex_text_to_bin(self, hex_plain_text: str):
        """ convert hexadecimal text to binary """
        bin_message = "".join(format(int(hex_plain_text[i:i+2], 16), '08b') for i in range(0, len(hex_plain_text), 2))
        return bin_message

    
    def convert_and_padd_message(self, plain_text: str):
        """ convert and padds message """

        # conversion to binary chain
        bin_message = self.convert_text_to_bin(plain_text)

        L = len(bin_message) // 8     # number of bytes computation
        X = 16 - L % 16               # number of element to pad
        bin_message += format(X, '08b') * X

        return bin_message
    

    def convert_to_text_and_unpadd(self, matrices_list):
        """ converts transformed matrix to plaintext and then unpads it """
        padded_plaintex = self.show_transformed_matrix_as_text(matrices_list, is_encryption= False)
        
        # padding removing : we know padding pattern covers 8bits = 1 text character
        # And we do not have more dans 16 padding characters due to the completion (X = 16 - L % 16)
        index, padding_character = len(padded_plaintex), padded_plaintex[-1]

        while padded_plaintex[index - 1] == padding_character:
            index -= 1

        return padded_plaintex[:index]
    

    def show_transformed_matrix_as_text(self, matrix_list, is_encryption = True):
        """ Displays the encrypted / decrypted message """
        message = ""
        for matrix in matrix_list:
            for i in range(4):
                message += "".join([format(elt, '02x') if is_encryption else chr(elt) for elt in matrix[:, i]])
        return message

    
    def build_4xN_matrix_from_32Nbits(self, block_32N_bits:str, N = 4):
        """ builds a 4xN matrix using 32*N bits block. 32*N = 128 or 192 or 256 """

        if len(block_32N_bits) != 32 * N:
            raise ValueError(f"The block needs to be {32 * N} bits block")   # block length control
        
        matrix_4xN = np.zeros((4, N), dtype=np.uint8)                  # matrix to build

        for i in range(0, len(block_32N_bits), 8):
            row, column = (i // 8) %4, i // 32                         # fillable by column
            matrix_4xN[row][column] = int(block_32N_bits[i:i+8], 2)
        
        return matrix_4xN
    

    def rot_word(self, word, n = -1):
        """ rotates a col-th W_{i-1} to the left """
        return np.roll(word.tolist(), n).ravel()   # makes left circular shitf
    

    def shift_rows_for_matrix(self, input_matrix_4x4, direction = 1):
        """ shifts row i, i times. is inplace operation """
        # direction = 1 for encryption and -1 for decryption
        for index in range(4):
            input_matrix_4x4[index, :] = self.rot_word(input_matrix_4x4[index, :], - direction * index)

    

    def sub_bytes(self, word, direction = 1):
        """ Apply S-Box transformation to a word  """
        output = []

        # determine SBox for encryption / decryption
        currentSBox = Sboxes_and_examples.Sbox_inv if direction == -1 else Sboxes_and_examples.Sbox
        
        for index, value in enumerate(word):
            hex_value = format(value, '02x')                #convesion of value of hexadecimal

            # Convert the first character to row and second to column
            row = int(hex_value[0], 16)  # First hex digit as row
            col = int(hex_value[1], 16)  # Second hex digit as column
            
            pos_in_SBox = (row << 4) | col                   # computation of position in sbox (as row * 2^4 + col)
            
            output.append(currentSBox[pos_in_SBox])
        
        return np.array(output).ravel()
    

    def sub_bytes_for_matrix(self, input_matrix_4x4, direction = 1):
        """ Apply S-Box transformation to a matrix. is inplace operation  """
        for index in range(4):
            input_matrix_4x4[index, :] = self.sub_bytes(input_matrix_4x4[index, :], direction)
    

    def r_con(self, index):
        """ returns the rcon word """
        
        if index >= len(self.Rcon):
            raise IndexError("rcon computation index is wrong")
        
        word = [0] * 4  # rcon word initialization
        word[0] = self.Rcon[index - 1]

        return np.array(word).ravel()
    

    def str_to_int_array(self, array: List[str]):
        """ converts string array to int array """
        return [int(x) for x in array]

    def polynomial_product(self, P: int, Q: int):
        """ computes the polynomial product of p1 and p2"""

        # conversion to binary array
        P = self.str_to_int_array(list(format(P, '08b')))
        Q = self.str_to_int_array(list(format(Q, '08b')))
        
        X8 = np.array([1, 1, 0, 1, 1])     # x^8 = x^4 + x^3 + x^1 + 1

        X = np.polymul(P, Q) % 2           # P * Q multiplication and modulo 2
        Result = np.zeros(8, dtype=int)    # results initialization

        while X.shape[0] > 8:              # we loop until multiplication as degree < 8
            R = X[-8:]                     # monomials with degree <= 7

            # bitwize operation
            Result = np.bitwise_xor(Result, R)

            P = X[:-8]                     # update of P
            Q = X8                         # update of Q

            # X computation
            X = np.polymul(P, Q) % 2
        
        # last R computation
        R = np.array([0] * (8 - len(X)) + X.tolist())
        Result = np.bitwise_xor(Result, R)
        
        return int("".join(Result.astype(str).tolist()), 2)
    

    def mix_columns(self, input_matrix, direction = 1):
        """ Performs the MixColumns operation in AES """
        multiplication = np.zeros_like(input_matrix)  # 4x4 0 matrix
        base_matrix = self.MM_Inv if direction == -1 else self.MM
        for line in range(4):
            for col in range(4):
                for k in range(4):
                    multiplication[line, col] ^= self.polynomial_product(base_matrix[line, k], input_matrix[k, col])
        return multiplication

### AES class

In [4]:
class AES:
    def __init__(self):
        self.message = None
        self.text_key = None
        self.keys_array = None
        self.utils = AESUtils()
        self.matrix_4x4_list = []
        self.transformed_matrix_list = None
    

    def set_message(self, message: str):
        """ sets the message that needs to be encrypted """
        self.message = message
    

    def set_key(self, key: str):
        """ sets the key that will be used for encryption """
        self.text_key = key
    

    def compute_4x4_matrices(self):
        """ computes 4x4 matrices """

        if not self.message:
            raise ValueError("message to encrypt/decrypt needs to me provided. use set_message mathod")
        
        # get padded binary message
        padded_bin_message = self.utils.convert_and_padd_message(self.message)

        self.matrix_4x4_list.clear()                                         # we make sure list is empty
        # loop to slice in 128bits
        for i in range(0, len(padded_bin_message), 128):
            slice_128 = padded_bin_message[i:i+128]                          # slicing
            matrix_4x4 = self.utils.build_4xN_matrix_from_32Nbits(slice_128) # 4x4 matrix computation (N = 4 by default)
            self.matrix_4x4_list.append(matrix_4x4)                          #append to 4x4  matrices list
    
    
    def compute_inverse_4x4_matrices(self):
        """ computes 4x4 matrices for decryption """

        if not self.message:
            raise ValueError("message to encrypt/decrypt needs to me provided. use set_message mathod")
        
        if len(self.message) % 32 != 0:
            raise ValueError("The provided cypher text has wrong length")
        
        self.matrix_4x4_list.clear()                                         # we make sure list is empty
        for i in range(0, len(self.message), 32):
            block = self.message[i:i+32]                                     # 32 bytes block (128 bits)
            bin_block = self.utils.convert_hex_text_to_bin(block)            # converts to 128 bits block
            matrix = self.utils.build_4xN_matrix_from_32Nbits(bin_block)     # matrix computing
            self.matrix_4x4_list.append(matrix)                              # add to list


    def key_expension(self):
        """ computes keys """
        if not self.text_key:
            raise ValueError("key is needed. use set_key method")
        
        key_in_bin = self.utils.convert_text_to_bin(self.text_key)
        N = len(key_in_bin) // 32         # number of 32-bit words in the key
        R = 7 + N                         # number of keys needed
        

        self.keys_array = np.zeros((4, 4 * R), dtype=np.uint8)  # matrice 4x(4R) where each columns is w_i

        # computation of the N first W_i (= K_i)
        W_4xN = self.utils.build_4xN_matrix_from_32Nbits(key_in_bin, N = N)

        # copy to key_array
        self.keys_array[:, :N] = W_4xN.copy()

        # computation of remaining W_i on the loop iteration
        for i in range(N, 4 * R):
            W_i_1 = self.keys_array[:, i - 1]                   # extraction of (i - 1)-th word
            W_i_N = self.keys_array[:, i - N].ravel()           # extraction of (i - N)-th word
            W_i = None

            # computing
            if i >= N and i % N == 0:                           # W_i = W_{i - N} xor SBox(Rotation(W_{i - 1})) xor Rcon_{i / N}          
                rotated = self.utils.rot_word(W_i_1)            # rotation
                sub_b = self.utils.sub_bytes(rotated)           # SBox operation
                r_con = self.utils.r_con(i // N)                # rcon computation
                
                W_i = W_i_N ^ sub_b ^ r_con                     # computation of W_i

            elif i >= N and N > 6 and i % N == 4:               # W_i = W_{i - N} xor SBox(W_{i - 1})
                sub_b = self.utils.sub_bytes(W_i_1)             # SBox operation
                
                W_i = W_i_N ^ sub_b                             # computation of W_i
            
            else:                                               # W_{i - N} xor W_{i - 1}
                W_i = W_i_N ^ W_i_1                             # computation of W_i
            
            self.keys_array[:, i] = W_i
    

    def add_round_key(self, message_4x4, key_4x4):
        """ Performs Add Round Key operation"""
        return np.bitwise_xor(message_4x4, key_4x4)
    
    
    def encrypt(self, message, key):
        """ Performs the encryption process """
        # we are in ECB mode
        self.transformed_matrix_list = []
        
        self.set_message(message)      # sets message
        self.set_key(key)              # sets key
        self.compute_4x4_matrices()    # computes 128 bits blocks as 4x4 8 bits matrix
        self.key_expension()           # expension of keys

        for matrix in self.matrix_4x4_list:
            state = matrix.copy()
            key = self.keys_array[:, :4]

            # first Add RoundKey
            state = self.add_round_key(state, key)
            
            # next rounds
            for round in range(4, self.keys_array.shape[1], 4):
                self.utils.sub_bytes_for_matrix(state)           # subBytes
                self.utils.shift_rows_for_matrix(state)          # shiftRows

                if round + 4 != self.keys_array.shape[1]:
                    state = self.utils.mix_columns(state)        # mixCOlumns
                
                key = self.keys_array[:, round:round + 4]
                state = self.add_round_key(state, key)           # AddRoundKey

            # 128 bits 4x4 matrix is encrypted > storing
            self.transformed_matrix_list.append(state)

        return self.utils.show_transformed_matrix_as_text(self.transformed_matrix_list)

    

    def decrypt(self, cyphertext, key):
        """ Performs the decryption process """
        # we are in ECB mode
        self.transformed_matrix_list = []
        
        self.set_message(cyphertext)                     # sets message
        self.set_key(key)                                # sets key
        self.compute_inverse_4x4_matrices()              # computes 128 bits blocks as 4x4 8 bits matrix
        self.key_expension()                             # expension of keys

        for matrix in self.matrix_4x4_list:
            state = matrix.copy()

            for i in range(self.keys_array.shape[1], 0, -4):

                key = self.keys_array[:, i-4:i]                                 # take last 4x4 matrix in the keys
                round = i // 4 - 1
                
                state = self.add_round_key(state, key)                          # first Add RoundKey
                
                if round > 0:                                                   # we apply shitf, subbytes and mix only if round > 0
                    if round != self.keys_array.shape[1] // 4 - 1:              # we apply mix-columns only if 0 < round < 10
                        state = self.utils.mix_columns(state, direction = -1)   # mix column inverse

                    self.utils.shift_rows_for_matrix(state, direction = -1)     # right shift of rows
                    
                    self.utils.sub_bytes_for_matrix(state, direction = -1)      # subBytes operation
                
            # add to transformed matrix list
            self.transformed_matrix_list.append(state)

        # conversion and unpadding
        plaintext = self.utils.convert_to_text_and_unpadd(self.transformed_matrix_list)

        return plaintext

## Execution of encrytion

In [5]:
# list of keys
keys = [Sboxes_and_examples.key, Sboxes_and_examples.key2, Sboxes_and_examples.key3]
message = Sboxes_and_examples.message                              # message to encrypt
cyphers = []                                                       # list to store cypher texts

aes = AES()                                                        # AES class instanciation

print(f"> Message to encrypt : {message}")
for key in keys:
    encrypted_message = aes.encrypt(message, key)
    cyphers.append(encrypted_message)
    print(f"(\n    + Key : {key}")
    print(f"    + Cypher text : {encrypted_message}\n)")

> Message to encrypt : Can you smell what the Rock is cooking?
(
    + Key : You can't see me
    + Cypher text : d69e09957672bb537f137948e9755d12ea924c80079da5b141a576d0142ed4c05c26547acb217669f3c0291966bafbe4
)
(
    + Key : Adrenaline in my soul   
    + Cypher text : 952291caeddd45a514214ef7c2e0bbd92ae6dfa584b6fe9d73acfbc2457fa73df725535f53e8b60d0c2121c9486d3edb
)
(
    + Key : Every thought out of control !!!
    + Cypher text : 4abe9fbb975f21ac88aba265147e437b47a1ae4b2b7b786d7bada2d50f1451cd582ac8988dfdec00d90f5104cb9eb5fb
)


## Execution of decrytion

In [6]:
aes = AES()

for cyphertext, key in zip(cyphers, keys):
    print(f"> Message to decrypt : {cyphertext}")
    plain_text = aes.decrypt(cyphertext, key)
    print(f"(\n    + Key : {key}")
    print(f"    + Plain text : {plain_text}\n)\n")

> Message to decrypt : d69e09957672bb537f137948e9755d12ea924c80079da5b141a576d0142ed4c05c26547acb217669f3c0291966bafbe4
(
    + Key : You can't see me
    + Plain text : Can you smell what the Rock is cooking?
)

> Message to decrypt : 952291caeddd45a514214ef7c2e0bbd92ae6dfa584b6fe9d73acfbc2457fa73df725535f53e8b60d0c2121c9486d3edb
(
    + Key : Adrenaline in my soul   
    + Plain text : Can you smell what the Rock is cooking?
)

> Message to decrypt : 4abe9fbb975f21ac88aba265147e437b47a1ae4b2b7b786d7bada2d50f1451cd582ac8988dfdec00d90f5104cb9eb5fb
(
    + Key : Every thought out of control !!!
    + Plain text : Can you smell what the Rock is cooking?
)

